In [1]:
import time
import pandas as pd
import numpy as np  # Import NumPy for numerical operations
import logger



## Step 1: Add a switch at the top to choose between PyTorch and PyPradie
use_pypradie = True  # Set this to True to use PyPradie instead of PyTorch

## Step 2: Import PyTorch or PyPradie dynamically
if use_pypradie:
    import pypradie as dl_framework  # Replace 'pypradie' with your actual PyPradie module
    library_used = "PyPradie"
else:
    import torch as dl_framework
    library_used = "PyTorch"

print(f"\nLibrary Used: {library_used}")

# Import necessary components based on the chosen framework
nn = dl_framework.nn
optim = dl_framework.optim
DataLoader = dl_framework.utils.data.DataLoader
TensorDataset = dl_framework.utils.data.TensorDataset
tensor = dl_framework.tensor
no_grad = dl_framework.no_grad

## Step 3: Load and Prepare the MNIST Dataset from CSV
train_df = pd.read_csv('../datasets/mnist/mnist_train.csv')
test_df = pd.read_csv('../datasets/mnist/mnist_test.csv')

print(train_df.shape)
print(test_df.shape)
# (60000, 785)
# (10000, 785)

# train_df = train_df.head(1000)
# test_df = test_df.head(100)

print(train_df.shape)
print(test_df.shape)

train_labels = train_df['label'].values
train_features = train_df.drop('label', axis=1).values / 255.0  # Normalize the pixel values
test_labels = test_df['label'].values
test_features = test_df.drop('label', axis=1).values / 255.0  # Normalize the pixel values

# Convert data to tensors using the dynamically chosen framework
train_features_tensor = tensor(train_features, dtype=dl_framework.float32)
train_labels_tensor = tensor(train_labels, dtype=dl_framework.long)
test_features_tensor = tensor(test_features, dtype=dl_framework.float32)
test_labels_tensor = tensor(test_labels, dtype=dl_framework.long)

batch_size = 64
train_dataset = TensorDataset(train_features_tensor, train_labels_tensor)
test_dataset = TensorDataset(test_features_tensor, test_labels_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Step 4: Define the Model (using nn directly, no dl_framework prefix needed)
model = nn.Sequential(
    nn.Linear(28 * 28, 128),  # Input layer (28x28 pixels = 784)
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

## Step 5: Define Loss Function and Optimizer (using SGD)
criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss from nn
optimizer = optim.SGD(model.parameters(), lr=0.01)  # SGD optimizer

## Step 6: Initialize a Dictionary to Track Training and Testing Information
tracking_data = {
    'epoch_count': 0,           # Tracks total number of epochs done
    'training_times': [],       # Time taken for each epoch
    'total_training_time': 0.0, # Total training time
    'testing_time': 0.0,        # Total time for testing
    'test_accuracy': 0.0,       # Accuracy on test data
    'train_accuracy': 0.0       # Accuracy on training data
}

## Step 7: Training Loop with Debugging
epochs = 5
for epoch in range(epochs):
    running_loss = 0.0
    correct_train = 0  # To track correct predictions for train accuracy
    total_train = 0
    start_time = time.time()  # Start timing the epoch
    
    for images, labels in train_loader:
        images = images.view(images.size(0), -1)  # Flatten the images into vectors
        
        optimizer.zero_grad()  # Zero the gradients
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        # Compute train accuracy
        predicted = dl_framework.argmax(outputs.detach(), dim=1)  # Using dl_framework's argmax
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    
    epoch_time = time.time() - start_time  # Calculate epoch duration
    tracking_data['epoch_count'] += 1  # Increment epoch counter
    tracking_data['training_times'].append(epoch_time)
    tracking_data['total_training_time'] += epoch_time
    
    train_accuracy = 100 * correct_train / total_train  # Train accuracy for this epoch
    tracking_data['train_accuracy'] = train_accuracy  # Update latest train accuracy

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {running_loss/len(train_loader):.6f}, Time: {epoch_time:.2f} seconds, Train Accuracy: {train_accuracy:.2f}%")

## Step 8: Testing the Model
correct_test = 0
total_test = 0

start_test_time = time.time()  # Start timing the testing phase

with no_grad():  # Disable gradient computation during testing
    for images, labels in test_loader:
        images = images.view(images.size(0), -1)  # Flatten images into vectors
        outputs = model(images)
        predicted = dl_framework.argmax(outputs, dim=1)  # Using dl_framework's argmax
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

# Calculate testing time and accuracy
tracking_data['testing_time'] = time.time() - start_test_time
tracking_data['test_accuracy'] = 100 * correct_test / total_test

## Step 9: Final Results
print(f"\nFinal Results:")
print(f"Test Accuracy: {tracking_data['test_accuracy']:.2f}%")
print(f"Train Accuracy: {tracking_data['train_accuracy']:.2f}%")
print(f"Testing Time: {tracking_data['testing_time']:.2f} seconds")
print(f"Total epochs: {tracking_data['epoch_count']}")
print(f"Total training time: {tracking_data['total_training_time']:.2f} seconds")
print(f"Average time per epoch: {sum(tracking_data['training_times']) / len(tracking_data['training_times']):.2f} seconds")
print(f"\nLibrary Used: {library_used}")



Library Used: PyPradie
(60000, 785)
(10000, 785)
(60000, 785)
(10000, 785)
Epoch 1/5, Loss: 0.895460, Time: 1.63 seconds, Train Accuracy: 75.79%
Epoch 2/5, Loss: 0.366198, Time: 1.25 seconds, Train Accuracy: 89.96%
Epoch 3/5, Loss: 0.302112, Time: 1.29 seconds, Train Accuracy: 91.52%
Epoch 4/5, Loss: 0.269531, Time: 1.40 seconds, Train Accuracy: 92.42%
Epoch 5/5, Loss: 0.245676, Time: 1.52 seconds, Train Accuracy: 93.08%

Final Results:
Test Accuracy: 93.38%
Train Accuracy: 93.08%
Testing Time: 0.05 seconds
Total epochs: 5
Total training time: 7.08 seconds
Average time per epoch: 1.42 seconds

Library Used: PyPradie


# PYTORCH

In [2]:
import time
import pandas as pd
import numpy as np  # Import NumPy for numerical operations
import logger



## Step 1: Add a switch at the top to choose between PyTorch and PyPradie
use_pypradie = False  # Set this to True to use PyPradie instead of PyTorch

## Step 2: Import PyTorch or PyPradie dynamically
if use_pypradie:
    import pypradie as dl_framework  # Replace 'pypradie' with your actual PyPradie module
    library_used = "PyPradie"
else:
    import torch as dl_framework
    library_used = "PyTorch"

print(f"\nLibrary Used: {library_used}")

# Import necessary components based on the chosen framework
nn = dl_framework.nn
optim = dl_framework.optim
DataLoader = dl_framework.utils.data.DataLoader
TensorDataset = dl_framework.utils.data.TensorDataset
tensor = dl_framework.tensor
no_grad = dl_framework.no_grad

## Step 3: Load and Prepare the MNIST Dataset from CSV
train_df = pd.read_csv('../datasets/mnist/mnist_train.csv')
test_df = pd.read_csv('../datasets/mnist/mnist_test.csv')

print(train_df.shape)
print(test_df.shape)
# (60000, 785)
# (10000, 785)

# train_df = train_df.head(1000)
# test_df = test_df.head(100)

print(train_df.shape)
print(test_df.shape)

train_labels = train_df['label'].values
train_features = train_df.drop('label', axis=1).values / 255.0  # Normalize the pixel values
test_labels = test_df['label'].values
test_features = test_df.drop('label', axis=1).values / 255.0  # Normalize the pixel values

# Convert data to tensors using the dynamically chosen framework
train_features_tensor = tensor(train_features, dtype=dl_framework.float32)
train_labels_tensor = tensor(train_labels, dtype=dl_framework.long)
test_features_tensor = tensor(test_features, dtype=dl_framework.float32)
test_labels_tensor = tensor(test_labels, dtype=dl_framework.long)

batch_size = 64
train_dataset = TensorDataset(train_features_tensor, train_labels_tensor)
test_dataset = TensorDataset(test_features_tensor, test_labels_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Step 4: Define the Model (using nn directly, no dl_framework prefix needed)
model = nn.Sequential(
    nn.Linear(28 * 28, 128),  # Input layer (28x28 pixels = 784)
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

## Step 5: Define Loss Function and Optimizer (using SGD)
criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss from nn
optimizer = optim.SGD(model.parameters(), lr=0.01)  # SGD optimizer

## Step 6: Initialize a Dictionary to Track Training and Testing Information
tracking_data = {
    'epoch_count': 0,           # Tracks total number of epochs done
    'training_times': [],       # Time taken for each epoch
    'total_training_time': 0.0, # Total training time
    'testing_time': 0.0,        # Total time for testing
    'test_accuracy': 0.0,       # Accuracy on test data
    'train_accuracy': 0.0       # Accuracy on training data
}

## Step 7: Training Loop with Debugging
epochs = 5
for epoch in range(epochs):
    running_loss = 0.0
    correct_train = 0  # To track correct predictions for train accuracy
    total_train = 0
    start_time = time.time()  # Start timing the epoch
    
    for images, labels in train_loader:
        images = images.view(images.size(0), -1)  # Flatten the images into vectors
        
        optimizer.zero_grad()  # Zero the gradients
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        # Compute train accuracy
        predicted = dl_framework.argmax(outputs.detach(), dim=1)  # Using dl_framework's argmax
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    
    epoch_time = time.time() - start_time  # Calculate epoch duration
    tracking_data['epoch_count'] += 1  # Increment epoch counter
    tracking_data['training_times'].append(epoch_time)
    tracking_data['total_training_time'] += epoch_time
    
    train_accuracy = 100 * correct_train / total_train  # Train accuracy for this epoch
    tracking_data['train_accuracy'] = train_accuracy  # Update latest train accuracy

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {running_loss/len(train_loader):.6f}, Time: {epoch_time:.2f} seconds, Train Accuracy: {train_accuracy:.2f}%")

## Step 8: Testing the Model
correct_test = 0
total_test = 0

start_test_time = time.time()  # Start timing the testing phase

with no_grad():  # Disable gradient computation during testing
    for images, labels in test_loader:
        images = images.view(images.size(0), -1)  # Flatten images into vectors
        outputs = model(images)
        predicted = dl_framework.argmax(outputs, dim=1)  # Using dl_framework's argmax
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

# Calculate testing time and accuracy
tracking_data['testing_time'] = time.time() - start_test_time
tracking_data['test_accuracy'] = 100 * correct_test / total_test

## Step 9: Final Results
print(f"\nFinal Results:")
print(f"Test Accuracy: {tracking_data['test_accuracy']:.2f}%")
print(f"Train Accuracy: {tracking_data['train_accuracy']:.2f}%")
print(f"Testing Time: {tracking_data['testing_time']:.2f} seconds")
print(f"Total epochs: {tracking_data['epoch_count']}")
print(f"Total training time: {tracking_data['total_training_time']:.2f} seconds")
print(f"Average time per epoch: {sum(tracking_data['training_times']) / len(tracking_data['training_times']):.2f} seconds")
print(f"\nLibrary Used: {library_used}")



Library Used: PyTorch
(60000, 785)
(10000, 785)
(60000, 785)
(10000, 785)
Epoch 1/5, Loss: 1.703799, Time: 2.18 seconds, Train Accuracy: 58.23%
Epoch 2/5, Loss: 0.557782, Time: 2.09 seconds, Train Accuracy: 85.36%
Epoch 3/5, Loss: 0.394624, Time: 2.15 seconds, Train Accuracy: 89.01%
Epoch 4/5, Loss: 0.345449, Time: 2.09 seconds, Train Accuracy: 90.16%
Epoch 5/5, Loss: 0.316129, Time: 2.08 seconds, Train Accuracy: 91.02%

Final Results:
Test Accuracy: 91.64%
Train Accuracy: 91.02%
Testing Time: 0.13 seconds
Total epochs: 5
Total training time: 10.59 seconds
Average time per epoch: 2.12 seconds

Library Used: PyTorch


Epoch 1/5, Loss: 0.895460, Time: 1.63 seconds, Train Accuracy: 75.79%
Epoch 2/5, Loss: 0.366198, Time: 1.25 seconds, Train Accuracy: 89.96%
Epoch 3/5, Loss: 0.302112, Time: 1.29 seconds, Train Accuracy: 91.52%
Epoch 4/5, Loss: 0.269531, Time: 1.40 seconds, Train Accuracy: 92.42%
Epoch 5/5, Loss: 0.245676, Time: 1.52 seconds, Train Accuracy: 93.08%

Final Results:
Test Accuracy: 93.38%
Train Accuracy: 93.08%
Testing Time: 0.05 seconds
Total epochs: 5
Total training time: 7.08 seconds
Average time per epoch: 1.42 seconds

Library Used: PyPradie


Epoch 1/5, Loss: 1.703799, Time: 2.18 seconds, Train Accuracy: 58.23%
Epoch 2/5, Loss: 0.557782, Time: 2.09 seconds, Train Accuracy: 85.36%
Epoch 3/5, Loss: 0.394624, Time: 2.15 seconds, Train Accuracy: 89.01%
Epoch 4/5, Loss: 0.345449, Time: 2.09 seconds, Train Accuracy: 90.16%
Epoch 5/5, Loss: 0.316129, Time: 2.08 seconds, Train Accuracy: 91.02%

Final Results:
Test Accuracy: 91.64%
Train Accuracy: 91.02%
Testing Time: 0.13 seconds
Total epochs: 5
Total training time: 10.59 seconds
Average time per epoch: 2.12 seconds

Library Used: PyTorch